# Session 8 — Building Smol Agents
## Focus: Develop autonomous agents that scaffold an entire codebase from a single prompt
### Key Concepts Introduction

**Explanation:** Smol Agents are minimalist AI systems that can generate complete codebases from natural language prompts. Unlike complex multi-agent systems, Smol Agents follow the "less is more" philosophy - doing more with simpler, focused architectures.
Installation & Setup

**Explanation:** Smol Agents represent a paradigm shift in AI-driven development by focusing on minimalist architectures that maximize output through intelligent prompt engineering and iterative refinement. Unlike traditional multi-agent systems that require complex coordination protocols, multiple specialized agents, and elaborate communication frameworks, Smol Agents embrace the philosophy of "doing more with less" by leveraging carefully crafted prompts and streamlined workflows. These systems are designed to understand high-level project requirements and translate them into fully functional codebases through a series of targeted generations and validations. The core innovation lies in their ability to maintain context across multiple generations, ensuring consistency in coding patterns, architecture decisions, and dependency management while operating with remarkably simple underlying architectures that are easier to debug, modify, and extend compared to their more complex counterparts.

In [1]:
# Install required packages
!pip install openai requests pathlib

  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.11.0-cp313-cp313-win_amd64.whl.metadata (5.3 kB)
  Using cached pydantic-2.11.9-py3-none-any.whl.metadata (68 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.33.2-cp313-cp313-win_amd64.whl.metadata (6.9 kB)
  Using cached typing_inspection-0.4.1-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/948.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/948.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/948.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/948.4 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/948.4 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/948.4 kB ? eta -:--:--
   ----------- 


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


**Explanation:** Set up OpenRouter configuration. OpenRouter provides a unified API for multiple AI models.

In [5]:
import os
import json
from pathlib import Path
from openai import OpenAI

# Configure OpenRouter instead of OpenAI
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key='sk-or-v1-e3ecd714bc83f6ee13f9d49fe97f29c1eb92c8b7987623b7a8db52a6df4ed438a'
)

## Understanding the Smol Developer Paradigm

**Explanation:** The Smol Developer paradigm represents a fundamental rethinking of how AI-assisted development should be structured, prioritizing simplicity, efficiency, and practical results over theoretical complexity. This approach is built on three interconnected principles that work together to create a highly effective code generation system. The first principle, "Single Prompt to Full Codebase," leverages advanced prompt engineering techniques to extract maximum contextual understanding from initial requirements, enabling the agent to generate coherent architectural plans before diving into individual file creation. The second principle, "Iterative Refinement," introduces intelligent feedback loops where the agent can analyze generation results, identify issues or inconsistencies, and automatically apply corrections through subsequent prompts. The third principle, "Minimal Dependencies," ensures that the generated codebases remain lightweight and maintainable by favoring native language features over external frameworks whenever possible, while also keeping the agent's own infrastructure simple enough to be understood and modified by individual developers.

## Building Your First Smol Agent with OpenRouter

**Explanation:** Constructing your initial Smol Agent with OpenRouter involves creating a flexible architecture that can adapt to different AI models while maintaining consistent output quality. The implementation typically begins with a core agent class that handles basic interactions with the OpenRouter API, including authentication, request formatting, and response parsing. This foundation is then enhanced with specialized methods for different aspects of code generation, such as project structure planning, file content creation, and dependency management. A key advantage of using OpenRouter is the ability to implement model-agnostic prompting strategies that produce reliable results across different AI providers, along with built-in fallback mechanisms that automatically switch to alternative models when primary options are unavailable or underperforming. The agent should also include comprehensive error handling for common API issues, rate limiting scenarios, and content filtering situations, ensuring graceful degradation rather than complete failure when encountering unexpected conditions.

In [29]:
class SmolDeveloper:
    def __init__(self, model="meta-llama/llama-3.1-8b-instruct"):  # OpenRouter model ID
        self.client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key='sk-or-v1-e3ecd714bc83f6ee13f9d49fe97f29c1eb92c8b7987623b7a8db52a6df4ed438a'
        )
        self.model = model

    def generate_file(self, filename, purpose, dependencies=None):
        """Generate a single file using OpenRouter AI"""
        prompt = f"""
        Create a complete, working {filename} file.
        Purpose: {purpose}
        {f"Dependencies: {dependencies}" if dependencies else ""}

        Requirements:
        - Include all necessary imports
        - Add comprehensive comments
        - Ensure the code is production-ready
        - Make it functional and testable

        Return ONLY the code without any explanations.
        """

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
        )

        return response.choices[0].message.content

    def create_project_structure(self, project_description):
        """Generate complete project structure using OpenRouter"""
        prompt = f"""
        Based on this project description: "{project_description}"

        Generate a JSON structure of files needed for this project.
       Return ONLY the JSON object without any additional text, markdown, or explanations.

        Required format:
        {{
            "project_name": "project_name_here",
            "files": [
                {{
                    "filename": "path/to/file.py",
                    "purpose": "description of what this file does"
                }}
            ]
        }}
        """

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
        )

        raw_content = response.choices[0].message.content.strip()

        # Remove any markdown code block markers if present
        json_content = raw_content.replace('```json', '').replace('```', '').strip()

        return json.loads(json_content)

## Hands-On: Generating a Hello World Web Application

**Explanation:** The "Hello World" web application exercise serves as a practical validation of your Smol Agent's capabilities while demonstrating the end-to-end workflow from prompt to deployed application. This process begins with a detailed prompt that specifies not just the basic functionality but also technical constraints, preferred frameworks, and deployment considerations. The agent first analyzes these requirements to generate an appropriate project structure, selecting optimal technologies based on the specified constraints and current best practices. It then proceeds to create each component systematically, starting with configuration files and build scripts, followed by core application logic, user interface elements, and finally testing and deployment configurations. Throughout this process, the agent maintains architectural consistency by ensuring that all generated components interoperate correctly and follow established patterns. The completed application should be immediately runnable and deployable, serving as both a functional product and a template for more complex projects.

In [11]:
# Initialize with OpenRouter model
developer = SmolDeveloper(model="meta-llama/llama-3.1-8b-instruct")

# Define our project
project_description = """
Create a modern 'Hello World' web application with:
- Flask backend API
- HTML/CSS/JavaScript frontend
- Requirements.txt for dependencies
- Comprehensive README with setup instructions
"""

# Generate the project structure using OpenRouter
print("📁 Generating project structure with OpenRouter...")
project_structure = developer.create_project_structure(project_description)
print(f"Project: {project_structure['project_name']}")

# Create the project directory
project_dir = Path(project_structure['project_name'])
project_dir.mkdir(exist_ok=True)

# Generate each file using OpenRouter AI
print("⚡ Generating files with OpenRouter...")
for file_info in project_structure['files']:
    file_path = project_dir / file_info['filename']
    file_path.parent.mkdir(parents=True, exist_ok=True)

    code = developer.generate_file(
        file_info['filename'],
        file_info['purpose']
    )

    file_path.write_text(code)
    print(f"✅ Created: {file_info['filename']}")

print(f"🎉 Project generated in: {project_dir}")

📁 Generating project structure with OpenRouter...
Project: hello_world
⚡ Generating files with OpenRouter...
✅ Created: requirements.txt
✅ Created: README.md
✅ Created: app.py
✅ Created: templates/index.html
✅ Created: static/style.css
✅ Created: static/script.js
🎉 Project generated in: hello_world


## Code Generation and Debugging Loops

**Explanation:** The integration of automated debugging loops represents a significant advancement in Smol Agent capabilities, transforming them from simple code generators into intelligent development partners. These loops work by systematically analyzing generated code for common issues such as syntax errors, logical inconsistencies, security vulnerabilities, and performance bottlenecks. When problems are detected, the agent formulates targeted correction prompts that address specific issues while preserving the overall code structure and functionality. The debugging process typically involves multiple validation stages, starting with basic syntax checking through language-specific tools, followed by static analysis for common patterns, and culminating with limited execution testing where safe and practical. This iterative approach not only produces higher-quality output but also helps the agent learn from its mistakes, gradually improving its understanding of what constitutes good code for different domains and use cases.

In [23]:
def debug_and_fix_code(developer, filepath, error_message):
    """Use OpenRouter to debug and fix code issues"""
    with open(filepath, 'r') as f:
        original_code = f.read()

    prompt = f"""
    Fix this code error:
    File: {filepath}
    Error: {error_message}

    Original code:
    {original_code}

    Provide the complete fixed code without explanations.
    """

    response = developer.client.chat.completions.create(
        model=developer.model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
    )

    fixed_code = response.choices[0].message.content
    return fixed_code

# Example debugging usage
try:
    # Try to run the generated code
    exec(open(project_dir / "app.py").read())
except Exception as e:
    print(f"🔧 Debugging error: {e}")
    fixed_code = debug_and_fix_code(developer, project_dir / "app.py", str(e))
    (project_dir / "app.py").write_text(fixed_code)
    print("✅ Code fixed!")

🔧 Debugging error: invalid syntax (<string>, line 1)
✅ Code fixed!


## Advanced Smol Agent Features

**Explanation:** Enhancing your Smol Agent with advanced features transforms it from a basic code generator into a sophisticated development assistant capable of handling complex, real-world scenarios. Multi-model support allows the agent to leverage different AI providers for specific tasks—using larger, more capable models for architectural planning while employing faster, more cost-effective models for routine file generation. Enhanced error handling goes beyond basic exception catching to include intelligent retry mechanisms, contextual error analysis, and automatic problem resolution strategies. Additional advanced features might include context-aware generation that maintains project-specific conventions, integration with version control systems for incremental updates, and collaboration features that enable multiple developers to work with the same agent instance. These enhancements collectively create a more robust and versatile tool that can adapt to varying project requirements and team workflows.

In [26]:
import os
from pathlib import Path
from openai import OpenAI
import shutil
from typing import List, Dict, Optional

class AdvancedSmolDeveloper:
    def __init__(self, model: str = "meta-llama/llama-3.1-8b-instruct"):
        self.client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key='sk-or-v1-e3ecd714bc83f6ee13f9d49fe97f29c1eb92c8b7987623b7a8db52a6df4ed438a'
        )
        self.model = model
        self.project_templates = self._initialize_templates()

    def _initialize_templates(self) -> Dict[str, List]:
        """Define project templates for different types of applications"""
        return {
            "flask_webapp": [
                ("app.py", "Main Flask application with routes and basic structure"),
                ("requirements.txt", "Python dependencies including Flask"),
                ("templates/index.html", "Main HTML template for the web interface"),
                ("static/css/style.css", "CSS styling for the web application"),
                ("config.py", "Configuration settings and environment variables"),
                ("models.py", "Database models if using SQLAlchemy"),
                ("README.md", "Comprehensive setup and deployment instructions")
            ],
            "cli_tool": [
                ("main.py", "Command-line interface with argument parsing"),
                ("requirements.txt", "Python dependencies for CLI functionality"),
                ("setup.py", "Package configuration for distribution"),
                ("README.md", "Usage examples and command reference")
            ],
            "data_analysis": [
                ("analysis.py", "Main data analysis script with pandas/numpy"),
                ("requirements.txt", "Data science libraries"),
                ("notebook.ipynb", "Jupyter notebook for exploratory analysis"),
                ("config.py", "Data source configurations and paths"),
                ("utils.py", "Helper functions for data processing")
            ]
        }

    def generate_file_content(self, filename: str, purpose: str, project_context: str) -> str:
        """Generate file content with detailed context"""
        prompt = f"""
        Create a complete, working {filename} file for a {project_context}.

        File purpose: {purpose}

        Requirements:
        - Include all necessary imports and dependencies
        - Add comprehensive comments explaining the code
        - Ensure the code follows best practices
        - Make it production-ready and functional
        - Include error handling where appropriate

        Return ONLY the code without any explanations or markdown formatting.
        """

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                max_tokens=2000
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"❌ Error generating {filename}: {e}")
            return f"# Error generating {filename}\n# Purpose: {purpose}\n\nprint('File generation failed')"

    def create_advanced_project(self, description: str, project_type: str = "flask_webapp",
                              custom_files: Optional[List] = None) -> Path:
        """Create an advanced project with template support and custom files"""

        # Generate project name from description
        project_name = description.lower().replace(' ', '_').replace('-', '_')[:30]
        project_dir = Path(project_name)

        # Clean existing directory or create new
        if project_dir.exists():
            shutil.rmtree(project_dir)
        project_dir.mkdir(exist_ok=True)

        print(f"🚀 Creating project: {project_name}")
        print(f"📁 Project type: {project_type}")
        print("=" * 50)

        # Get file list from template or custom files
        if custom_files:
            files_to_create = custom_files
        else:
            files_to_create = self.project_templates.get(project_type, [])
            if not files_to_create:
                print(f"⚠️  Project type '{project_type}' not found. Using default template.")
                files_to_create = [
                    ("app.py", "Main application file"),
                    ("requirements.txt", "Python dependencies"),
                    ("README.md", "Project documentation")
                ]

        # Generate each file
        for filename, purpose in files_to_create:
            file_path = project_dir / filename

            # Create directory structure if needed
            file_path.parent.mkdir(parents=True, exist_ok=True)

            print(f"📄 Generating: {filename}")
            code = self.generate_file_content(filename, purpose, description)

            # Write file
            file_path.write_text(code)
            print(f"✅ Created: {filename}")

        # Generate project documentation
        self._generate_project_summary(project_dir, description, files_to_create)

        print("=" * 50)
        print(f"🎉 Project successfully created: {project_dir}")
        return project_dir

    def _generate_project_summary(self, project_dir: Path, description: str, files: List):
        """Generate a project summary file"""
        summary_content = f"""# Project Summary

## Description
{description}

## Generated Files
"""
        for filename, purpose in files:
            summary_content += f"- **{filename}**: {purpose}\n"

        summary_content += f"""
## Total Files Generated
{len(files)} files

## Next Steps
1. Review the generated code
2. Install dependencies: `pip install -r requirements.txt`
3. Run the application
4. Customize as needed
"""

        summary_file = project_dir / "PROJECT_SUMMARY.md"
        summary_file.write_text(summary_content)

    def add_file_to_project(self, project_path: Path, filename: str, purpose: str):
        """Add a new file to an existing project"""
        if not project_path.exists():
            print(f"❌ Project directory {project_path} does not exist")
            return

        file_path = project_path / filename
        file_path.parent.mkdir(parents=True, exist_ok=True)

        print(f"📄 Adding new file: {filename}")
        code = self.generate_file_content(filename, purpose, "existing project")
        file_path.write_text(code)
        print(f"✅ Added: {filename}")

    def regenerate_file(self, project_path: Path, filename: str, new_purpose: str = None):
        """Regenerate a specific file with updated purpose"""
        file_path = project_path / filename

        if not file_path.exists():
            print(f"❌ File {filename} does not exist in project")
            return

        # Read original file to understand context
        original_content = file_path.read_text()

        purpose = new_purpose or f"Updated version of {filename}"
        print(f"🔄 Regenerating: {filename}")

        new_code = self.generate_file_content(filename, purpose, "project improvement")
        file_path.write_text(new_code)
        print(f"✅ Regenerated: {filename}")

# Enhanced usage examples
def demonstrate_advanced_features():
    """Showcase the advanced features"""
    advanced_dev = AdvancedSmolDeveloper()

    # Example 1: Create a Flask web application
    print("Example 1: Creating Flask Web Application")
    flask_project = advanced_dev.create_advanced_project(
        "A task management web application with user authentication and database",
        project_type="flask_webapp"
    )

    # Example 2: Add additional files to existing project
    print("\nExample 2: Adding Additional Files")
    advanced_dev.add_file_to_project(
        flask_project,
        "auth.py",
        "User authentication and authorization logic"
    )

    # Example 3: Create a CLI tool
    print("\nExample 3: Creating CLI Tool")
    cli_project = advanced_dev.create_advanced_project(
        "A command-line tool for file management and batch processing",
        project_type="cli_tool"
    )

    return flask_project, cli_project

# Run the advanced demonstration
if __name__ == "__main__":
    flask_proj, cli_proj = demonstrate_advanced_features()

    print(f"\n📊 Projects created:")
    print(f"- Flask Web App: {flask_proj}")
    print(f"- CLI Tool: {cli_proj}")

Example 1: Creating Flask Web Application
🚀 Creating project: a_task_management_web_applicat
📁 Project type: flask_webapp
📄 Generating: app.py
✅ Created: app.py
📄 Generating: requirements.txt
✅ Created: requirements.txt
📄 Generating: templates/index.html
✅ Created: templates/index.html
📄 Generating: static/css/style.css
✅ Created: static/css/style.css
📄 Generating: config.py
✅ Created: config.py
📄 Generating: models.py
✅ Created: models.py
📄 Generating: README.md
✅ Created: README.md
🎉 Project successfully created: a_task_management_web_applicat

Example 2: Adding Additional Files
📄 Adding new file: auth.py
✅ Added: auth.py

Example 3: Creating CLI Tool
🚀 Creating project: a_command_line_tool_for_file_m
📁 Project type: cli_tool
📄 Generating: main.py
✅ Created: main.py
📄 Generating: requirements.txt
✅ Created: requirements.txt
📄 Generating: setup.py
✅ Created: setup.py
📄 Generating: README.md
✅ Created: README.md
🎉 Project successfully created: a_command_line_tool_for_file_m

📊 Projects

## Case Study: Rapid MVP Prototyping Assistant

**Explanation:** The Rapid MVP Prototyping Assistant case study demonstrates how Smol Agents can dramatically accelerate the initial phases of product development by automating the creation of functional prototypes from high-level concept descriptions. This specialized agent goes beyond basic code generation to include user story mapping, database schema design, API endpoint planning, and UI/UX considerations. It begins by analyzing the product concept to identify core features, user flows, and technical requirements, then generates a complete implementation plan that balances functionality with development complexity. The agent selectively uses OpenRouter's diverse model portfolio—employing more creative models for ideation phases and more structured models for implementation details. The resulting MVP includes not just working code but also documentation, testing suites, and deployment configurations, providing teams with a solid foundation for further development rather than just a proof-of-concept demonstration.

In [27]:
import os
from pathlib import Path
from openai import OpenAI
import shutil
from typing import List, Dict, Optional, Tuple

class AdvancedSmolDeveloper:
    def __init__(self, model: str = "meta-llama/llama-3.1-8b-instruct"):
        self.client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key='sk-or-v1-e3ecd714bc83f6ee13f9d49fe97f29c1eb92c8b7987623b7a8db52a6df4ed438a'
        )
        self.model = model
        self.available_models = [
            "meta-llama/llama-3.1-8b-instruct",
            "google/gemini-pro",
            "anthropic/claude-3-sonnet"
        ]
        self.project_templates = self._initialize_templates()

    def _initialize_templates(self) -> Dict[str, List]:
        """Define project templates for different types of applications"""
        return {
            "flask_webapp": [
                ("app.py", "Main Flask application with routes and basic structure"),
                ("requirements.txt", "Python dependencies including Flask"),
                ("templates/index.html", "Main HTML template for the web interface"),
                ("static/css/style.css", "CSS styling for the web application"),
                ("config.py", "Configuration settings and environment variables"),
                ("models.py", "Database models if using SQLAlchemy"),
                ("README.md", "Comprehensive setup and deployment instructions")
            ],
            "cli_tool": [
                ("main.py", "Command-line interface with argument parsing"),
                ("requirements.txt", "Python dependencies for CLI functionality"),
                ("setup.py", "Package configuration for distribution"),
                ("README.md", "Usage examples and command reference")
            ],
            "data_analysis": [
                ("analysis.py", "Main data analysis script with pandas/numpy"),
                ("requirements.txt", "Data science libraries"),
                ("notebook.ipynb", "Jupyter notebook for exploratory analysis"),
                ("config.py", "Data source configurations and paths"),
                ("utils.py", "Helper functions for data processing")
            ],
            "fastapi_api": [
                ("main.py", "FastAPI application with endpoints"),
                ("requirements.txt", "Python dependencies including FastAPI and uvicorn"),
                ("schemas.py", "Pydantic schemas for request/response validation"),
                ("database.py", "Database connection and models"),
                ("README.md", "API documentation and usage examples")
            ]
        }

    def generate_with_fallback(self, prompt: str, primary_model: str = None) -> Tuple[str, str]:
        """Generate content with model fallback using OpenRouter"""
        models_to_try = [primary_model] if primary_model else self.available_models

        for model in models_to_try:
            try:
                response = self.client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.1,
                    max_tokens=2000
                )
                return response.choices[0].message.content.strip(), model
            except Exception as e:
                print(f"Model {model} failed: {e}")
                continue

        raise Exception("All models failed")

    def generate_file_content(self, filename: str, purpose: str, project_context: str) -> str:
        """Generate file content with detailed context"""
        prompt = f"""
        Create a complete, working {filename} file for a {project_context}.

        File purpose: {purpose}

        Requirements:
        - Include all necessary imports and dependencies
        - Add comprehensive comments explaining the code
        - Ensure the code follows best practices
        - Make it production-ready and functional
        - Include error handling where appropriate

        Return ONLY the code without any explanations or markdown formatting.
        """

        content, model_used = self.generate_with_fallback(prompt)
        return content

    def create_advanced_project(self, description: str, project_type: str = "flask_webapp",
                              custom_files: Optional[List] = None) -> Path:
        """Create an advanced project with template support and custom files"""

        # Generate project name from description
        project_name = description.lower().replace(' ', '_').replace('-', '_')[:30]
        project_dir = Path(project_name)

        # Clean existing directory or create new
        if project_dir.exists():
            shutil.rmtree(project_dir)
        project_dir.mkdir(exist_ok=True)

        print(f"🚀 Creating project: {project_name}")
        print(f"📁 Project type: {project_type}")
        print("=" * 50)

        # Get file list from template or custom files
        if custom_files:
            files_to_create = custom_files
        else:
            files_to_create = self.project_templates.get(project_type, [])
            if not files_to_create:
                print(f"⚠️  Project type '{project_type}' not found. Using default template.")
                files_to_create = [
                    ("app.py", "Main application file"),
                    ("requirements.txt", "Python dependencies"),
                    ("README.md", "Project documentation")
                ]

        # Generate each file
        for filename, purpose in files_to_create:
            file_path = project_dir / filename

            # Create directory structure if needed
            file_path.parent.mkdir(parents=True, exist_ok=True)

            print(f"📄 Generating: {filename}")
            code = self.generate_file_content(filename, purpose, description)

            # Write file
            file_path.write_text(code)
            print(f"✅ Created: {filename}")

        # Generate project documentation
        self._generate_project_summary(project_dir, description, files_to_create)

        print("=" * 50)
        print(f"🎉 Project successfully created: {project_dir}")
        return project_dir

    def _generate_project_summary(self, project_dir: Path, description: str, files: List):
        """Generate a project summary file"""
        summary_content = f"""# Project Summary

## Description
{description}

## Generated Files
"""
        for filename, purpose in files:
            summary_content += f"- **{filename}**: {purpose}\n"

        summary_content += f"""
## Total Files Generated
{len(files)} files

## Next Steps
1. Review the generated code
2. Install dependencies: `pip install -r requirements.txt`
3. Run the application
4. Customize as needed
"""

        summary_file = project_dir / "PROJECT_SUMMARY.md"
        summary_file.write_text(summary_content)

    def add_file_to_project(self, project_path: Path, filename: str, purpose: str):
        """Add a new file to an existing project"""
        if not project_path.exists():
            print(f"❌ Project directory {project_path} does not exist")
            return

        file_path = project_path / filename
        file_path.parent.mkdir(parents=True, exist_ok=True)

        print(f"📄 Adding new file: {filename}")
        code = self.generate_file_content(filename, purpose, "existing project")
        file_path.write_text(code)
        print(f"✅ Added: {filename}")

    def regenerate_file(self, project_path: Path, filename: str, new_purpose: str = None):
        """Regenerate a specific file with updated purpose"""
        file_path = project_path / filename

        if not file_path.exists():
            print(f"❌ File {filename} does not exist in project")
            return

        purpose = new_purpose or f"Updated version of {filename}"
        print(f"🔄 Regenerating: {filename}")

        new_code = self.generate_file_content(filename, purpose, "project improvement")
        file_path.write_text(new_code)
        print(f"✅ Regenerated: {filename}")

    def analyze_and_improve_project(self, project_path: Path):
        """Analyze project structure and suggest improvements"""
        if not project_path.exists():
            print(f"❌ Project directory {project_path} does not exist")
            return

        # Get list of files in project
        files = []
        for file_path in project_path.rglob("*"):
            if file_path.is_file():
                relative_path = file_path.relative_to(project_path)
                files.append(str(relative_path))

        prompt = f"""
        Analyze this project structure and suggest improvements:
        Project files: {files}

        Provide specific suggestions for:
        1. Missing files that would improve the project
        2. Code quality improvements
        3. Security considerations
        4. Performance optimizations

        Return a concise list of recommendations.
        """

        analysis, model_used = self.generate_with_fallback(prompt)
        print("🔍 Project Analysis Results:")
        print(analysis)
        return analysis

class MVPGenerator:
    def __init__(self, model: str = "meta-llama/llama-3.1-8b-instruct"):
        self.developer = AdvancedSmolDeveloper(model)

    def generate_mvp(self, idea: str, features: str, timeline: str = "1 week") -> Path:
        """Generate a complete MVP from an idea using OpenRouter"""
        # First, determine the best project type based on the idea
        project_type = self._determine_project_type(idea, features)

        # Create custom file list based on features
        custom_files = self._create_custom_file_list(idea, features, project_type)

        # Generate the project
        description = f"{idea} with features: {features}. Timeline: {timeline}"
        project_path = self.developer.create_advanced_project(
            description=description,
            project_type=project_type,
            custom_files=custom_files
        )

        # Generate MVP-specific documentation
        self._generate_mvp_documentation(project_path, idea, features, timeline)

        return project_path

    def _determine_project_type(self, idea: str, features: str) -> str:
        """Determine the best project template based on the idea and features"""
        if any(keyword in idea.lower() + features.lower() for keyword in ['web', 'website', 'browser', 'ui']):
            return "flask_webapp"
        elif any(keyword in idea.lower() + features.lower() for keyword in ['api', 'rest', 'endpoint']):
            return "fastapi_api"
        elif any(keyword in idea.lower() + features.lower() for keyword in ['cli', 'command', 'terminal']):
            return "cli_tool"
        elif any(keyword in idea.lower() + features.lower() for keyword in ['data', 'analysis', 'pandas', 'numpy']):
            return "data_analysis"
        else:
            return "flask_webapp"  # Default

    def _create_custom_file_list(self, idea: str, features: str, project_type: str) -> List[Tuple[str, str]]:
        """Create a custom file list tailored to the MVP features"""
        base_files = self.developer.project_templates.get(project_type, [])

        # Add feature-specific files
        custom_files = base_files.copy()

        if 'authentication' in features.lower():
            custom_files.append(("auth.py", "User authentication and authorization system"))
            custom_files.append(("templates/login.html", "User login page"))

        if 'database' in features.lower():
            custom_files.append(("models.py", "Database models and schemas"))
            custom_files.append(("database.py", "Database connection and utilities"))

        if 'api' in features.lower():
            custom_files.append(("api.py", "API routes and endpoints"))
            custom_files.append(("schemas.py", "Request/response schemas"))

        return custom_files

    def _generate_mvp_documentation(self, project_path: Path, idea: str, features: str, timeline: str):
        """Generate MVP-specific documentation"""
        mvp_doc = f"""# MVP Documentation

## Idea
{idea}

## Key Features
{features}

## Development Timeline
{timeline}

## MVP Scope
This is a Minimum Viable Product focusing on core functionality.

## Next Development Steps
1. Test core functionality
2. Gather user feedback
3. Plan next feature iterations
4. Consider scalability improvements

## Success Metrics
- Core functionality working
- Basic user interface functional
- Key features implemented
- Ready for user testing
"""

        doc_file = project_path / "MVP_README.md"
        doc_file.write_text(mvp_doc)
        print(f"📋 MVP documentation created: {doc_file}")

# Advanced usage examples
def demonstrate_advanced_features():
    """Showcase the advanced features"""
    print("=== ADVANCED SMOL DEVELOPER DEMONSTRATION ===")

    # Example 1: Create a Flask web application using MVP Generator
    print("\n1. Creating MVP with MVPGenerator")
    mvp_gen = MVPGenerator()
    mvp_project = mvp_gen.generate_mvp(
        "AI-powered content summarizer",
        "URL input, text extraction, AI summarization, export options",
        "2 weeks"
    )

    # Example 2: Use AdvancedSmolDeveloper directly for custom project
    print("\n2. Creating custom project with AdvancedSmolDeveloper")
    advanced_dev = AdvancedSmolDeveloper()

    custom_files = [
        ("main.py", "Primary application logic with advanced features"),
        ("utils/helpers.py", "Utility functions and helpers"),
        ("tests/test_basic.py", "Unit tests for core functionality"),
        ("docs/usage.md", "Comprehensive usage documentation"),
        ("Dockerfile", "Container configuration for deployment"),
        ("docker-compose.yml", "Multi-container orchestration"),
        (".env.example", "Environment variables template"),
        ("scripts/deploy.sh", "Deployment automation script")
    ]

    custom_project = advanced_dev.create_advanced_project(
        "Enterprise-grade microservice with testing and deployment",
        custom_files=custom_files
    )

    # Example 3: Analyze and improve an existing project
    print("\n3. Analyzing and improving project")
    advanced_dev.analyze_and_improve_project(mvp_project)

    # Example 4: Add features to existing project
    print("\n4. Adding features to existing project")
    advanced_dev.add_file_to_project(
        mvp_project,
        "export.py",
        "Data export functionality in multiple formats"
    )

    return mvp_project, custom_project

# Run the advanced demonstration
if __name__ == "__main__":
    try:
        mvp_proj, custom_proj = demonstrate_advanced_features()

        print(f"\n📊 Projects created successfully:")
        print(f"- MVP Project: {mvp_proj}")
        print(f"- Custom Project: {custom_proj}")
        print(f"\n🎯 Advanced features demonstrated:")
        print("- MVP generation with feature analysis")
        print("- Custom project templates")
        print("- Project analysis and improvement")
        print("- Dynamic file addition")

    except Exception as e:
        print(f"❌ Error during demonstration: {e}")

# Quick individual usage examples
def quick_examples():
    """Quick individual usage examples"""
    print("\n=== QUICK USAGE EXAMPLES ===")

    # Quick MVP generation
    mvp_gen = MVPGenerator()
    quick_mvp = mvp_gen.generate_mvp(
        "Task management app",
        "User authentication, task CRUD operations, due dates",
        "1 week"
    )

    # Quick custom project
    dev = AdvancedSmolDeveloper()
    quick_project = dev.create_advanced_project(
        "Data analysis tool for sales data",
        project_type="data_analysis"
    )

    return quick_mvp, quick_project


=== ADVANCED SMOL DEVELOPER DEMONSTRATION ===

1. Creating MVP with MVPGenerator
🚀 Creating project: ai_powered_content_summarizer_
📁 Project type: flask_webapp
📄 Generating: app.py
✅ Created: app.py
📄 Generating: requirements.txt
✅ Created: requirements.txt
📄 Generating: templates/index.html
✅ Created: templates/index.html
📄 Generating: static/css/style.css
✅ Created: static/css/style.css
📄 Generating: config.py
✅ Created: config.py
📄 Generating: models.py
✅ Created: models.py
📄 Generating: README.md
✅ Created: README.md
🎉 Project successfully created: ai_powered_content_summarizer_
📋 MVP documentation created: ai_powered_content_summarizer_\MVP_README.md

2. Creating custom project with AdvancedSmolDeveloper
🚀 Creating project: enterprise_grade_microservice_
📁 Project type: flask_webapp
📄 Generating: main.py
✅ Created: main.py
📄 Generating: utils/helpers.py
✅ Created: utils/helpers.py
📄 Generating: tests/test_basic.py
✅ Created: tests/test_basic.py
📄 Generating: docs/usage.md
✅ Creat

# Best Practices for Smol Agents

**Explanation:** Key principles for effective smol agent development using OpenRouter:
### Prompt Engineering

Effective prompt engineering is the cornerstone of successful Smol Agent operations, requiring careful consideration of how requirements are formulated and presented to AI models. Specificity in prompts involves not just describing what needs to be built but also outlining technical constraints, performance requirements, coding standards, and integration points with existing systems. Including constraints helps guide the AI toward appropriate technology selections and architectural patterns while avoiding overly complex or impractical solutions. The iterative aspect of prompt engineering recognizes that the first generation is rarely perfect—developers should plan for multiple refinement cycles where initial results are analyzed, specific issues are identified, and targeted follow-up prompts are crafted to address deficiencies. This process benefits greatly from maintaining detailed generation histories that track which prompts produced which results, enabling continuous improvement of prompt strategies over time.

### Error Handling

Robust error handling in Smol Agent implementations requires a multi-layered approach that anticipates various failure modes and provides appropriate responses for each. Model fallbacks leverage OpenRouter's diverse model portfolio to maintain service availability even when primary models experience issues, with intelligent routing that considers factors like cost, latency, and task appropriateness. Validation processes should include both automated checks (syntax validation, dependency resolution, basic static analysis) and human-reviewable outputs that make it easy to spot problems before code is executed. Graceful degradation planning ensures that partial successes are still valuable—for example, if an agent cannot generate a complete application, it should still deliver usable components with clear documentation about what remains to be implemented. This approach minimizes disruption and maintains productivity even when facing unexpected challenges.

### Project Structure

Thoughtful project structure generation is critical for creating maintainable, scalable codebases that can evolve over time. Modular design principles ensure that generated components have clear responsibilities, well-defined interfaces, and minimal dependencies on other parts of the system. This approach facilitates testing, debugging, and future modifications by localizing changes to specific modules. Comprehensive documentation should be integrated throughout the generation process, including not just high-level README files but also code comments, API documentation, configuration guides, and deployment instructions. Configuration management makes projects adaptable to different environments and use cases by externalizing settings, supporting multiple deployment scenarios, and providing clear extension points for customizations. Together, these structural considerations transform generated code from disposable prototypes into sustainable foundations for long-term development.

In [30]:
# Final comprehensive smol agent example
def create_complete_project(requirements, model="meta-llama/llama-3.1-8b-instruct"):
    """End-to-end project generation with OpenRouter"""
    developer = SmolDeveloper(model=model)

    # Plan the project
    structure = developer.create_project_structure(requirements)

    # Generate all files
    for file_info in structure['files']:
        code = developer.generate_file(file_info['filename'], file_info['purpose'])
        Path(file_info['filename']).write_text(code)

    # Generate setup instructions
    setup_guide = developer.generate_file(
        "SETUP.md",
        "Comprehensive setup and installation guide"
    )

    return structure['project_name']

# Generate a complete project
project_name = create_complete_project(
    "A REST API for a blog with CRUD operations, authentication, and file uploads",
    "meta-llama/llama-3.1-8b-instruct"
)
print(f"🎯 Project {project_name} created successfully!")

🎯 Project Blog API created successfully!
